In [61]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated,Literal
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from dotenv import load_dotenv
import os
from IPython.display import Image
from pydantic import BaseModel,Field
from langchain_groq import ChatGroq
import operator

In [62]:
load_dotenv()

True

In [63]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

In [64]:
class SentimentSchema(TypedDict):
    sentiment: Literal["positive","negative"]=Field(description="Sentiment of the text.")


In [65]:
class DiagnosisSchema(TypedDict):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [66]:
structured_model=llm.with_structured_output(SentimentSchema)
structured_model2 = llm.with_structured_output(DiagnosisSchema)

In [67]:
text="I had a wonderful day at the park ith my friends."
prompt=f"What is the sentiment of the  following text {text}"
res=structured_model.invoke(prompt)
print(res)

{'sentiment': 'positive'}


In [68]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive","negative"]
    diagnosis: dict
    response: str

In [69]:
def find_sentiment(state:ReviewState):
    prompt=f"For the given text find out the sentiment \n {state["review"]}"
    sentiment=structured_model.invoke(prompt)

    return {'sentiment': sentiment['sentiment']}

def check_sentiment(state:ReviewState)->Literal["positive_response","run_diagnosis"]:
    if state['sentiment']=='positive':
        return "positive_response"
    else:
        return "run_diagnosis"

def positive_response(state:ReviewState):
    prompt=f"Write a thank-you message in response to the review {state['review']} and also ask the user to leave feedbak on the website."
    res=llm.invoke(prompt)
    return {'response':res.content}

def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response}

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = llm.invoke(prompt).content

    return {'response': response}

In [70]:
graph=StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow=graph.compile()

In [71]:
initial_state={
    'review':"Worst product that I have ever used."
}
res=workflow.invoke(initial_state)
print(res)

{'review': 'Worst product that I have ever used.', 'sentiment': 'negative', 'diagnosis': {'issue_type': 'Bug', 'tone': 'angry', 'urgency': 'high'}, 'response': "I'm so sorry to hear that you're experiencing a bug issue and that it's causing frustration for you. I can imagine how annoying it must be, and I'm here to help resolve the problem as quickly as possible.\n\nI've marked your issue as high priority, and I'm committed to finding a solution for you. Can you please provide me with more details about the bug, such as the steps you took leading up to the issue and any error messages you've received? This will help me to better understand the problem and work towards a resolution.\n\nIn the meantime, I want to assure you that I'm doing everything I can to get this issue fixed for you. If there's a temporary workaround that can help alleviate the problem, I'll be happy to provide that to you as well.\n\nYour satisfaction is my top priority, and I appreciate your patience and cooperatio

In [73]:
print(res['review'])
print(res['sentiment'])
print(res['diagnosis'])
print(res['response'])

Worst product that I have ever used.
negative
{'issue_type': 'Bug', 'tone': 'angry', 'urgency': 'high'}
I'm so sorry to hear that you're experiencing a bug issue and that it's causing frustration for you. I can imagine how annoying it must be, and I'm here to help resolve the problem as quickly as possible.

I've marked your issue as high priority, and I'm committed to finding a solution for you. Can you please provide me with more details about the bug, such as the steps you took leading up to the issue and any error messages you've received? This will help me to better understand the problem and work towards a resolution.

In the meantime, I want to assure you that I'm doing everything I can to get this issue fixed for you. If there's a temporary workaround that can help alleviate the problem, I'll be happy to provide that to you as well.

Your satisfaction is my top priority, and I appreciate your patience and cooperation as we work through this issue together. Please know that I'm 